<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

# Phase 1 — SCAMPS Augmented Adaptation

Start from `checkpoints/best_pretrained/best.pth` (FP_PURE).
Train on SCAMPS 2000 subjects with video-only augmentations to strip CG-domain shortcuts.
Per-epoch composite score on UBFC + MCD-IriunWebcam held-out + rPPG-10.
Best checkpoint saved at `checkpoints/phase1/best.pth`.

**Composite score baseline (FP_PURE):** 1.0000
Exit gate: score ≤ 1.0 (no regression from FP_PURE).


In [1]:
import sys, os, json, time, gc, math, warnings
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd
import h5py
import cv2
import torch
import torch.nn as nn
from scipy.signal import butter, filtfilt, find_peaks
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/mnt/sata-ssd/rppg_project')
FP_ROOT      = PROJECT_ROOT / 'external' / 'FactorizePhys'
sys.path.insert(0, str(FP_ROOT))

CACHE_DIR    = PROJECT_ROOT / 'rppg_dataset' / 'SCAMPS' / 'scamps_cache_72_y5f_f16'
SPLIT_CSV    = PROJECT_ROOT / 'rppg_dataset' / 'SCAMPS' / 'ScampsTrainTestSplit.csv'
UBFC_CACHE   = Path('/tmp/ubfc_y5f_clips_72.pt')
RPPG10_ROOT  = PROJECT_ROOT / 'rppg_dataset' / 'rPPG-10' / 'Dataset_rPPG-10'
MCD_ROOT     = PROJECT_ROOT / 'rppg_dataset' / 'MCD-rPPG'
MCD_DB_CSV   = MCD_ROOT / 'db.csv'
MCD_VID_DIR  = MCD_ROOT / 'video'
MCD_SPLIT    = PROJECT_ROOT / 'checkpoints' / 'mcd_split.json'

START_CKPT   = PROJECT_ROOT / 'checkpoints' / 'best_pretrained' / 'best.pth'
OUT_DIR      = PROJECT_ROOT / 'checkpoints' / 'phase1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT    = OUT_DIR / 'best.pth'
LAST_CKPT    = OUT_DIR / 'last.pth'
METRICS_JSON = OUT_DIR / 'metrics.json'
RPPG10_CACHE = OUT_DIR / 'rppg10_eval_cache.pt'
MCD_CACHE    = OUT_DIR / 'mcd_eval_cache.pt'
TRAIN_SCRIPT = PROJECT_ROOT / 'src' / 'train_phase1.py'
CFG_PATH     = OUT_DIR / 'config.json'

# ── Hyperparams ───────────────────────────────────────────────────────────────
EPOCHS         = 15
BATCH_SIZE     = 16    # per GPU (total 32 across 2 GPUs)
CLIP_LEN       = 160
IMG_SIZE       = 72
CLIPS_PER_SUBJ = 4
LR_MAX         = 3e-4
LR_MIN         = 1e-6
WEIGHT_DECAY   = 1e-4
GRAD_CLIP      = 1.0
NUM_WORKERS    = 4
SEED           = 42
WORLD_SIZE     = torch.cuda.device_count()

MCD_CAM           = 'IriunWebcam'
MCD_CLIPS_PER_VID = 2
RPPG10_CLIPS_SUBJ = 12

assert START_CKPT.exists(), f'Missing: {START_CKPT}'
assert UBFC_CACHE.exists(), f'Missing: {UBFC_CACHE}'
print(f'CUDA devices: {WORLD_SIZE}')
print(f'Start ckpt:   {START_CKPT}')
print(f'UBFC cache:   {UBFC_CACHE} ({UBFC_CACHE.stat().st_size/1e9:.2f} GB)')


CUDA devices: 2
Start ckpt:   /mnt/sata-ssd/rppg_project/checkpoints/best_pretrained/best.pth
UBFC cache:   /tmp/ubfc_y5f_clips_72.pt (4.84 GB)


In [2]:
from datetime import datetime
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M')
CLEARML_TASK_ID = None

try:
    from clearml import Task
    task = Task.init(
        project_name='rppg',
        task_name=f'phase1/run_{RUN_DATE}',
        reuse_last_task_id=False,
    )
    CLEARML_TASK_ID = task.id
    task.set_parameters({
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
        'lr_max': LR_MAX, 'lr_min': LR_MIN,
        'weight_decay': WEIGHT_DECAY, 'grad_clip': GRAD_CLIP,
        'clips_per_subj': CLIPS_PER_SUBJ, 'mcd_cam': MCD_CAM,
    })
    print(f'ClearML task id: {CLEARML_TASK_ID}')
except Exception as e:
    print(f'ClearML unavailable: {e}')
    task = None


ClearML Task: created new task id=8055f2b62e1540c48a56eadf33e7f2f3


2026-05-24 00:04:21,163 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


ClearML results page: https://app.clear.ml/projects/d74f4bfc8cd64ac7bfd6e79389891452/experiments/8055f2b62e1540c48a56eadf33e7f2f3/output/log


ClearML task id: 8055f2b62e1540c48a56eadf33e7f2f3


In [3]:
# ── Load SCAMPS split ─────────────────────────────────────────────────────────
df = pd.read_csv(SPLIT_CSV, names=['subject_id', 'url', 'split'], skiprows=1)
train_ids = df[df['split'] == 'Train']['subject_id'].tolist()
val_ids   = df[df['split'] == 'Val']['subject_id'].tolist()
print(f'SCAMPS: {len(train_ids)} train / {len(val_ids)} val subjects')

# ── Helper: HR extraction ─────────────────────────────────────────────────────
def extract_hr_fft(ppg, fps=30.0, lo=0.6, hi=4.0):
    freqs = np.fft.rfftfreq(len(ppg), d=1.0 / fps)
    fft   = np.abs(np.fft.rfft(ppg))
    mask  = (freqs >= lo) & (freqs <= hi)
    if not mask.any(): return float('nan')
    return freqs[mask][np.argmax(fft[mask])] * 60.0


def extract_hr_ecg(ecg, fs=1000.0):
    nyq = fs / 2.0
    b, a = butter(4, [5.0 / nyq, 20.0 / nyq], btype='bandpass')
    filt = filtfilt(b, a, ecg.astype('float64'))
    sq   = filt ** 2
    peaks, _ = find_peaks(sq, distance=int(0.3 * fs), height=np.percentile(sq, 75))
    if len(peaks) < 2: return 75.0
    rr = np.diff(peaks) / fs
    rr = rr[(rr >= 0.3) & (rr <= 2.0)]
    return float(60.0 / rr.mean()) if len(rr) > 0 else 75.0


SCAMPS: 2000 train / 400 val subjects


In [4]:
# ── rPPG-10 eval cache ────────────────────────────────────────────────────────
def build_rppg10_cache(root, clip_len, clips_per_subj, target_size, out_path):
    if out_path.exists():
        data = torch.load(str(out_path), weights_only=False)
        print(f'rPPG-10 cache loaded: {len(data)} clips')
        return data

    print('Building rPPG-10 eval cache...')
    cache = []
    for subj_dir in tqdm(sorted(root.glob('Subject_*')), desc='rPPG-10'):
        name     = subj_dir.name
        vid_path = subj_dir / f'{name}_Cheek1_.avi'
        ecg_path = subj_dir / f'{name}_ECG.npy'
        if not vid_path.exists() or not ecg_path.exists():
            continue

        cap    = cv2.VideoCapture(str(vid_path))
        fps_v  = cap.get(cv2.CAP_PROP_FPS) or 30.0
        frames = []
        while True:
            ret, fr = cap.read()
            if not ret: break
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            fr = cv2.resize(fr, (target_size, target_size), interpolation=cv2.INTER_AREA)
            frames.append(fr)
        cap.release()
        if len(frames) < clip_len + 1: continue

        ecg       = np.load(str(ecg_path))
        ecg_fs    = 1000.0
        max_start = len(frames) - clip_len - 1
        starts    = [int(i * max_start / max(clips_per_subj - 1, 1))
                     for i in range(clips_per_subj)]

        for s in starts:
            clip_t = torch.from_numpy(
                np.stack(frames[s: s + clip_len + 1]).astype(np.float32)
                .transpose(3, 0, 1, 2) / 255.0)
            ecg_s = int(s / fps_v * ecg_fs)
            ecg_e = int((s + clip_len) / fps_v * ecg_fs)
            seg   = ecg[ecg_s:ecg_e] if ecg_e <= len(ecg) else ecg[-int(clip_len / fps_v * ecg_fs):]
            gt_hr = extract_hr_ecg(seg, ecg_fs) if len(seg) > int(0.6 * ecg_fs) else float('nan')
            cache.append({'frames': clip_t, 'subj': name, 'gt_hr': gt_hr})

    torch.save(cache, str(out_path))
    print(f'rPPG-10 cache saved: {len(cache)} clips → {out_path}')
    return cache


# ── MCD eval cache ────────────────────────────────────────────────────────────
def _yolo_detect(model, frame_rgb, coef=1.5):
    h, w = frame_rgb.shape[:2]
    res  = model.detect_face(frame_rgb)
    if res is None: return [0, 0, w, h]
    x1, y1, x2, y2 = res
    sq  = max(x2 - x1, y2 - y1)
    cx  = (x1 + x2) // 2; cy = (y1 + y2) // 2
    ex  = max(0, cx - int(coef * sq / 2))
    ey  = max(0, cy - int(coef * sq / 2))
    return [ex, ey, int(coef * sq), int(coef * sq)]


def build_mcd_cache(mcd_root, split_json, db_csv, camera, clip_len,
                    clips_per_vid, target_size, out_path):
    if out_path.exists():
        data = torch.load(str(out_path), weights_only=False)
        print(f'MCD cache loaded: {len(data)} clips')
        return data

    print('Building MCD eval cache (YOLO5Face on CPU)...')
    from dataset.data_loader.face_detector.YOLO5Face import YOLO5Face
    yolo = YOLO5Face(backend='Y5F', device='cpu')

    with open(split_json) as f:
        split = json.load(f)
    held_out = set(int(x) for x in split.get('held_out', split.get('test', [])))

    db      = pd.read_csv(db_csv)
    vid_dir = mcd_root / 'video'
    cache   = []

    for pid in tqdm(sorted(held_out), desc='MCD subjects'):
        for step in ['before', 'after']:
            vp = vid_dir / f'{pid}_{camera}_{step}.avi'
            if not vp.exists(): continue
            rows  = db[(db['patient_id'] == pid) & (db['step'] == step)]
            gt_hr = float(rows['pulse'].mean()) if len(rows) > 0 else float('nan')
            if not (40 <= gt_hr <= 200): continue

            cap   = cv2.VideoCapture(str(vp))
            ret0, f0 = cap.read()
            if not ret0: cap.release(); continue
            f0_rgb = cv2.cvtColor(f0, cv2.COLOR_BGR2RGB)
            bbox   = _yolo_detect(yolo, f0_rgb)
            bx, by, bw, bh = bbox
            sq     = max(bw, bh)

            frames = [f0]
            while True:
                ret, fr = cap.read()
                if not ret: break
                frames.append(fr)
            cap.release()
            if len(frames) < clip_len + 1: continue

            max_start = len(frames) - clip_len - 1
            starts    = [int(i * max_start / max(clips_per_vid - 1, 1))
                         for i in range(clips_per_vid)]

            for s in starts:
                clip_frames = []
                for t in range(s, s + clip_len + 1):
                    fr   = cv2.cvtColor(frames[t], cv2.COLOR_BGR2RGB)
                    crop = fr[max(0,by):max(0,by)+sq, max(0,bx):max(0,bx)+sq]
                    if crop.size == 0: crop = fr
                    clip_frames.append(
                        cv2.resize(crop, (target_size, target_size),
                                   interpolation=cv2.INTER_AREA))
                arr   = np.stack(clip_frames).astype(np.float32) / 255.0
                clip_t = torch.from_numpy(arr.transpose(3, 0, 1, 2))
                cache.append({'frames': clip_t, 'subj': str(pid),
                              'step': step, 'gt_hr': gt_hr})

    torch.save(cache, str(out_path))
    print(f'MCD cache saved: {len(cache)} clips → {out_path}')
    return cache


# ── Build all caches ──────────────────────────────────────────────────────────
print('=== Building eval caches (one-time pre-processing) ===')
t0 = time.time()
rppg10_cache = build_rppg10_cache(RPPG10_ROOT, CLIP_LEN, RPPG10_CLIPS_SUBJ, IMG_SIZE, RPPG10_CACHE)
mcd_cache    = build_mcd_cache(MCD_ROOT, MCD_SPLIT, MCD_DB_CSV, MCD_CAM, CLIP_LEN,
                               MCD_CLIPS_PER_VID, IMG_SIZE, MCD_CACHE)
ubfc_cache   = torch.load(str(UBFC_CACHE), weights_only=False)
print(f'Cache build done ({time.time()-t0:.0f}s)')
print(f'Eval clips — UBFC: {len(ubfc_cache)} | rPPG-10: {len(rppg10_cache)} | MCD: {len(mcd_cache)}')


=== Building eval caches (one-time pre-processing) ===
Building rPPG-10 eval cache...


rPPG-10:   0%|          | 0/27 [00:00<?, ?it/s]

rPPG-10 cache saved: 312 clips → /mnt/sata-ssd/rppg_project/checkpoints/phase1/rppg10_eval_cache.pt
Building MCD eval cache (YOLO5Face on CPU)...


I0000 00:00:1779545097.121868   71689 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


MCD subjects:   0%|          | 0/100 [00:00<?, ?it/s]

...

[mpeg4 @ 0x3160d0c0] ac-tex damaged at 27 2
[mpeg4 @ 0x3160d0c0] Error at MB: 109


[mpeg4 @ 0x29e53100] ac-tex damaged at 29 2
[mpeg4 @ 0x29e53100] Error at MB: 111


[mpeg4 @ 0xbf721880] mcbpc damaged at 25 25
[mpeg4 @ 0xbf721880] Error at MB: 1050


MCD cache saved: 400 clips → /mnt/sata-ssd/rppg_project/checkpoints/phase1/mcd_eval_cache.pt


Cache build done (223s)
Eval clips — UBFC: 483 | rPPG-10: 312 | MCD: 400


In [5]:
# ── Verify training script ───────────────────────────────────────────────────
assert TRAIN_SCRIPT.exists(), f'Training script missing: {TRAIN_SCRIPT}'
print(f'Training script: {TRAIN_SCRIPT} ({TRAIN_SCRIPT.stat().st_size/1024:.0f} KB)')

# ── Write config JSON ─────────────────────────────────────────────────────────
cfg = {
    'cache_dir':       str(CACHE_DIR),
    'ubfc_cache':      str(UBFC_CACHE),
    'rppg10_cache':    str(RPPG10_CACHE),
    'mcd_cache':       str(MCD_CACHE),
    'start_ckpt':      str(START_CKPT),
    'best_ckpt':       str(BEST_CKPT),
    'last_ckpt':       str(LAST_CKPT),
    'metrics_json':    str(METRICS_JSON),
    'epochs':          EPOCHS,
    'batch_size':      BATCH_SIZE,
    'clip_len':        CLIP_LEN,
    'img_size':        IMG_SIZE,
    'clips_per_subj':  CLIPS_PER_SUBJ,
    'lr_max':          LR_MAX,
    'lr_min':          LR_MIN,
    'weight_decay':    WEIGHT_DECAY,
    'grad_clip':       GRAD_CLIP,
    'num_workers':     NUM_WORKERS,
    'seed':            SEED,
    'train_ids':       train_ids,
    'clearml_task_id': CLEARML_TASK_ID,
}
CFG_PATH.write_text(json.dumps(cfg, indent=2))
print(f'Config saved: {CFG_PATH}')
print(f'Train IDs: {len(train_ids)} subjects × {CLIPS_PER_SUBJ} clips = '
      f'{len(train_ids)*CLIPS_PER_SUBJ} clips/epoch')
print(f'Steps/epoch per GPU: {len(train_ids)*CLIPS_PER_SUBJ // (BATCH_SIZE*WORLD_SIZE)}')

Training script: /mnt/sata-ssd/rppg_project/src/train_phase1.py (19 KB)
Config saved: /mnt/sata-ssd/rppg_project/checkpoints/phase1/config.json
Train IDs: 2000 subjects × 4 clips = 8000 clips/epoch
Steps/epoch per GPU: 250


In [6]:
# ── Launch DDP training ───────────────────────────────────────────────────────
# This cell runs training via torchrun and streams output here.
# Training will block until all epochs complete.

import subprocess, sys

cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    f'--nproc_per_node={WORLD_SIZE}',
    '--standalone',
    str(TRAIN_SCRIPT),
]
print('Command:', ' '.join(cmd))
print()
print('Starting training...')
print('=' * 60)
t0 = time.time()

result = subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    capture_output=False,  # stream stdout/stderr directly
)

elapsed = time.time() - t0
if result.returncode == 0:
    print(f'\nTraining complete. Elapsed: {elapsed/60:.1f} min')
else:
    print(f'\nTraining FAILED (returncode={result.returncode}). Elapsed: {elapsed/60:.1f} min')


Command: /home/dex/rppg_venv/bin/python -m torch.distributed.run --nproc_per_node=2 --standalone /mnt/sata-ssd/rppg_project/src/train_phase1.py

Starting training...


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


Train clips: 8000 | Steps/epoch: 250 | Batch: 16×2=32


Evaluating starting checkpoint...


Start — UBFC=10.55 MCD=15.59 rPPG10=17.41 score=3.4535


Epoch 1/15:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1/15:   0%|          | 0/250 [00:13<?, ?it/s]
[rank0]: Traceback (most recent call last):
[rank0]:   File "/mnt/sata-ssd/rppg_project/src/train_phase1.py", line 455, in <module>
[rank0]:     main()
[rank0]:   File "/mnt/sata-ssd/rppg_project/src/train_phase1.py", line 389, in main
[rank0]:     loss    = neg_pearson(pn, ln) + 0.2 * freq_loss(pn, ln)
[rank0]:                                           ^^^^^^^^^^^^^^^^^
[rank0]:   File "/mnt/sata-ssd/rppg_project/src/train_phase1.py", line 262, in freq_loss
[rank0]:     pf    = torch.abs(torch.fft.rfft(pred,  dim=-1))[:, mask]
[rank0]:                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[rank0]: RuntimeError: cuFFT only supports dimensions whose sizes are powers of two when computing in half precision, but got a signal size of[160]
[rank1]: Traceback (most recent call last):
[rank1]:   File "/mnt/sata-ssd/rppg_project/src/train_phase1.py", line 455, in <module>
[rank1]:     main()
[rank1]:   File "/mnt/sata-ssd/rppg_project/src/train

W0524 00:11:05.097000 77159 torch/distributed/elastic/multiprocessing/api.py:897] Sending process 77176 closing signal SIGTERM
E0524 00:11:05.197000 77159 torch/distributed/elastic/multiprocessing/api.py:869] failed (exitcode: -9) local_rank: 0 (pid: 77175) of binary: /home/dex/rppg_venv/bin/python
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/dex/rppg_venv/lib/python3.12/site-packages/torch/distributed/run.py", line 923, in <module>
    main()
  File "/home/dex/rppg_venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 355, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/home/dex/rppg_venv/lib/python3.12/site-packages/torch/distributed/run.py", line 919, in main
    run(args)
  File "/home/dex/rppg_venv/lib/python3.12/site-packages/torch/distributed/run.py", line 910, in run
    elastic_launch(
  Fil

           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dex/rppg_venv/lib/python3.12/site-packages/torch/distributed/launcher/api.py", line 269, in launch_agent
    raise ChildFailedError(
torch.distributed.elastic.multiprocessing.errors.ChildFailedError: 
/mnt/sata-ssd/rppg_project/src/train_phase1.py FAILED
------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-05-24_00:11:05
  host      : Desktop-Dex
  rank      : 0 (local_rank: 0)
  exitcode  : -9 (pid: 77175)
  error_file: <N/A>
  traceback : Signal 9 (SIGKILL) received by PID 77175



Training FAILED (returncode=1). Elapsed: 2.9 min


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
# ── Final report ─────────────────────────────────────────────────────────────
with open(str(METRICS_JSON)) as f:
    history = json.load(f)

epochs   = [r['epoch']       for r in history]
losses   = [r['train_loss']  for r in history]
ubfc_m   = [r['ubfc_mae']    for r in history]
mcd_m    = [r['mcd_mae']     for r in history]
r10_m    = [r['rppg10_mae']  for r in history]
scores   = [r['score']       for r in history]
best_idx = int(np.argmin(scores))
best     = history[best_idx]

print('Phase 1 — Results')
print('=' * 72)
print(f'{'Ep':>3} | {'Loss':>8} | {'UBFC':>6} | {'MCD':>6} | {'rPPG10':>7} | {'Score':>7}')
print('-' * 72)
for r in history:
    mark = '★' if r['epoch'] == best['epoch'] else ' '
    print(f'{mark}{r["epoch"]:2d} | {r["train_loss"]:8.5f} | '
          f'{r["ubfc_mae"]:6.2f} | {r["mcd_mae"]:6.2f} | '
          f'{r["rppg10_mae"]:7.2f} | {r["score"]:7.4f}')
print('=' * 72)
print(f'\nBest epoch    : {best["epoch"]}')
print(f'Composite score: {best["score"]:.4f}  (baseline = 1.0000)')
print(f'  UBFC  MAE  : {best["ubfc_mae"]:.2f} bpm  (baseline 1.23)')
print(f'  MCD   MAE  : {best["mcd_mae"]:.2f} bpm  (baseline 12.36)')
print(f'  rPPG10 MAE : {best["rppg10_mae"]:.2f} bpm  (baseline 13.89)')
print(f'\nCheckpoint: {BEST_CKPT}')
print()
if best['score'] <= 1.0:
    print('EXIT GATE: PASSED  (score ≤ 1.0 — no regression from FP_PURE)')
    print('Phase 2 starting point:', BEST_CKPT)
else:
    print('EXIT GATE: FAILED  (score > 1.0 — regression from FP_PURE)')
    print('Phase 2 starting point: checkpoints/best_pretrained/best.pth (FP_PURE)')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

axs[0].plot(epochs, losses, 'b-o', ms=4)
axs[0].set(title='Train Loss', xlabel='Epoch', ylabel='Loss')

axs[1].plot(epochs, ubfc_m, 'C0-o', ms=4, label='UBFC')
axs[1].plot(epochs, mcd_m,  'C1-s', ms=4, label='MCD')
axs[1].plot(epochs, r10_m,  'C2-^', ms=4, label='rPPG-10')
for base, col in [(1.23,'C0'), (12.36,'C1'), (13.89,'C2')]:
    axs[1].axhline(base, color=col, ls='--', alpha=0.5)
axs[1].set(title='HR MAE per Dataset', xlabel='Epoch', ylabel='MAE (bpm)')
axs[1].legend()

axs[2].plot(epochs, scores, 'r-o', ms=4)
axs[2].axhline(1.0, color='k', ls='--', label='Baseline')
axs[2].axvline(best['epoch'], color='g', ls=':', alpha=0.7, label='Best')
axs[2].set(title='Composite Score', xlabel='Epoch', ylabel='Score')
axs[2].legend()

plt.tight_layout()
plot_path = OUT_DIR / 'phase1_training_curves.png'
plt.savefig(str(plot_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot: {plot_path}')


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/sata-ssd/rppg_project/checkpoints/phase1/metrics.json'

In [ ]:
# ── Update execution log ─────────────────────────────────────────────────────
from datetime import date
today = date.today().isoformat()

phase2_start = (str(BEST_CKPT) if best['score'] <= 1.0
                else str(PROJECT_ROOT / 'checkpoints' / 'best_pretrained' / 'best.pth'))
gate = 'PASSED' if best['score'] <= 1.0 else 'FAILED (regression)'

exec_log = PROJECT_ROOT / 'docs' / 'execution_log.md'
entry = (
    f'\n## {today}\n'
    f'**Phase 1 — SCAMPS Augmented Adaptation — {gate}**\n'
    f'- Epochs: {EPOCHS}, best epoch: {best["epoch"]}\n'
    f'- Composite score: {best["score"]:.4f} (baseline 1.0000)\n'
    f'- UBFC: {best["ubfc_mae"]:.2f} | MCD: {best["mcd_mae"]:.2f} | '
    f'rPPG-10: {best["rppg10_mae"]:.2f}\n'
    f'- Checkpoint: {BEST_CKPT}\n'
    f'- Phase 2 starts from: {phase2_start}\n'
)
with open(exec_log, 'a') as f:
    f.write(entry)
print(f'Appended to {exec_log}')
print(entry)
